Eventually, you'll want to run a computational workflow for a given system.  

To do so, you need: 

0) To **configure scope** in a computation cluster (i.e. a cluster with a job manager, SLURM or SGE). You can also configure scope in your local computer, to analyze the data, but you won't be able to submit computations. 
 
    To do it, you can run the script "**scope_config**" in either/both the local computer and the computation cluster, and follow instructions.

    ```bash
    scope_config -h
    ```

    This will create a environment-class object that will contain all the relevant information for the job submission. 

1) To **create a system** and add sources (molecules, or unit cells).  

2) To **create a job file**, as a plain text file, for easier interaction. 

3) Then, you can run SCOPE in a computation cluster (i.e. a cluster with a job manager, SLURM or SGE), with one -or many- job files, and a given system. To do so, you can use the "scope_run_job" script in the computation cluster: 

    ```bash
    scope_run_job -h
    ```

In [ ]:
import os
import scope

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data/Tutorial_4/"
tutorial_folder = os.path.abspath('.')+'/Data/Tutorial_4/'

# Example:

## 1.1) Create a System

In [ ]:
## In this example, we create a simple system: water
from scope.classes_system import System
test_sys   = System("water")

In [ ]:
## Then, it is recommended to set up the paths of the three relevant folders for Calculations (attribute self.calcs_path), Sources (self.sources_path), and System (self.sys_path)
## You can do it in 3 ways:

## (1) Directly as: 
test_sys.sources_path = f"{tutorial_folder}Sources/water/"
test_sys.calcs_path   = f"{tutorial_folder}Calcs/water/"
test_sys.sys_path     = f"{tutorial_folder}Systems/water/"
test_sys.sys_file     = f"{tutorial_folder}Systems/water/water.npy"

## (2) Reading from Environment-class object (uncomment to use). If you're working on a notebook, it might be more difficult to use:
# First, you load (or create) the enviroment
#   env = load_binary(path) 
# Second:
#   self.set_paths_from_environment(env)

## (3) With self.set_paths(), you just need to follow the instructions. If you're working on a notebook, it might be more difficult to use

In [ ]:
## Once done, you can print the system to check that everything is correct
print(test_sys)

## You can check whether the paths are correct with. It will also return TRUE/FALSE if folders exist: 
print(test_sys.check_paths(debug=1))

## Normally, WARNINGS should appear, but don't worry for the moment

## 1.2) Create a Molecule and Associate it to the system

### 1.2.1) The basics

In [ ]:
## Now we create an actual water molecule, whose geometry is read from an .xyz file
from scope.classes_specie import Molecule
from scope.read_write import read_xyz

labels, coords = read_xyz(test_sys.sources_path+"water.xyz")
molec = molecule(labels, coords)
print(molec)

### 1.2.2) Charge and Spin (See also Tutorial 0)

In [ ]:
## By default, molecules are neutral (and singlets) when created. 
## You can change it if needed:
molec.set_total_charge(2)
## Notice that the charge has changed:
molec

In [ ]:
## molec.set_total_charge() and molec.set_total_spin() are enough if you just want to specify charge and multiplicity for a quantum chemistry computation.
## These functions just set the charge to the first atom of the molecule
molec.atoms[0] 

## Notice that we have here a H atom with +2 charge (!)

In [ ]:
## It is much better if you individually select the atom you want to put the charge in, like:
molec.reset_charge()
molec.atoms[1].set_charge(-1)
molec

## Notice that we reset the charge, and now added a -1 charge to the O atom, which makes slightly more sense. 
## SPECIE and CELL charges/spins are always computed as the sum of atomic charges/spins hosted in its ATOMS.
## This means that, if you change an ATOM charge/spin, you're automatically changing the charge of the parent class 

In [ ]:
## For SPIN, it works very similarly: 
molec.set_spin_metals(2, debug=1)
## Logically, it won't find any metal in water, and it will complain, but this is something that should work in a TMC
## I recommend you to continue tutorial 0 and try some of these options in a CELL and SPECIE sources of ABITEM

### 1.2.3) Initial State

In [ ]:
## Now we Create The Initial State, from which we will draw the starting data for the computations.
ini_state = molec.add_state("initial")
## As we mention in Tutorial 2, "initial" states are not created automatically. The reason is that the information they require depends on the source type.
## SPECIE sources only require geometry: 
ini_state.set_geometry(labels, coords)
## CELL sources would also require the cell_vectors and/or cell_parameters, which we don't have here:
#ini_state.set_cell() 

## 1.3) Finish and Save the system

In [ ]:
## And source it to our system, giving it the name "reference"
_ = test_sys.add_source('reference',molec)
## Notice that the molecule has been added:
print(test_sys)

In [ ]:
## Trying to add new molecules with the same name won't work, to avoid duplicities
_ = test_sys.add_source('reference',molec, debug=1)

In [ ]:
## Finally, save the system to a file. By default, it will use test_sys.sys_file
test_sys.save()

## 2) Create a Job File

In [ ]:
from scope.classes_input import *

In [ ]:
## Job Specs can be created from a file, or from a string. 
## This is an example input read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable

test_input = """
&environment
   max_jobs         = 80
   max_procs        = 240
   requested_procs  = 8
/

&options
/

&job_data
   branch       = 'Isolated'
   workflow     = ['reference']
   job_setup    = 'regular'
   suffix       = 'opt'
   keyword      = 'pbe opt'

   istate       = 'initial'
   fstate       = 'pbe opt'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = False
/

&qc_data
   software     = 'g16'
   jobtype      = 'opt'
   functional   = 'pbe'
   basis        = 'def2SVP'
   is_grimme    = True
   loose_opt    = True
/"""

In [ ]:
# IMPORTANT. Notice that, when running the 'scope_run_job' script. The job file (-j arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from scope.read_write import save_text
file_name = ''.join([tutorial_folder,"test_job.job"])
save_text(test_input, file_name)

In [ ]:
## In any case, the job can be interpreted and stored as INPUT_DATA class objects:
## There are 4 input sections. Which are read separately 
local_env   = Input_data(content=test_input, section="&environment", isfile=False)
options     = Input_data(content=test_input, section="&options", isfile=False)
job_data    = Input_data(content=test_input, section="&job_data", isfile=False)
qc_data     = Input_data(content=test_input, section="&qc_data", isfile=False)

## They could be read in a single line (see below), but the data would not be stored correctly 
## inp = input_data(content=test_input, isfile=False)  

### Section 1: Local Environment

In [ ]:
# Here are the options related to the computation resources, that are not hardcoded during the environment creation. 
# The user can change these options anytime in the job file, without having to modify the environment
local_env

In [ ]:
# There are some defaults as well, which are always added to the user options
local_env = fill_environment_data(local_env)
print(local_env)

In [ ]:
## Notice that a "method" variable has been added. This one specifies how best queue is chosen if more than one is available

### Section 2: Options

In [ ]:
## These are other options related to the submission:
options = fill_options_data(options)
print(options)

## This section was empty in the "test_input", but now defaults were added:

# want_submit:  
    # Here, the user can turn off submission (want_submit = False) of jobs. 
    # If so, input files will be created, but computations won't be submitted. 
    # This is perfect for testing the input files creation

# ignore_submitted: 
    # Normally when submitting a computation, SCOPE verifies that no computation with the same name is already running. 
    # If activated (ignore_submitted = True), this check won't be done, and multiple instances of the same job will be created.
    # This might be OK in some cases, when those job, even if they have the same name, refer to different input files.

# overwrite_inputs and overwrite_logs:
    # When true, the code can overwrite files that already exist.

### Section 3: Job Data Section

In [ ]:
## This section includes job information. 
# 
## Basically, a Job with this information will be created (if it doesn't exist yet) or found (if it already exists) inside each system's workflow
## and other related information: 
job_data = fill_job_data(job_data)
print(job_data)

In [ ]:
## Here's the meaning of each variable:

# branch:       to locate the job inside the system, the BRANCH must be specified. 
# workflow:     to locate the job inside the system, the WORKFLOW must be specified. If more than one, one job will be created inside each workflow
# setup:        how are COMPUTATIONS inside the job created
# suffix:       which string must be added to the input/output/sub files to identify. For example, in ABITEM_opt_r1_HS.log, "opt" is the suffix
# keyword:      a name given to this job. 
# istate:       state from which the information must be extracted 
# fstate:       state where the information will be stored
# hierarchy:    expected order of jobs inside workflow
# requisites:   list of keywords of jobs that should have finished before this one can start
# constrains:   list of keywords of jobs that should have NOT finished before this one can start
# must_be_good: whether the job can considered done even if it did not finish correctly. 
    # if must_be_good = true, it will try hard to do the task. For instance, if the user requested an optimization, it will not stop until reaching the minimum
    # if must_be_good = false, it will perform just one attempt

### Section 4: QC data

In [ ]:
## Finally, we have the information of the actual Quantum Chemistry (QC) computation.
qc_data = fill_qc_data(qc_data)
print(qc_data)

## This is the information that will be passed to the QC software to create the input 

## In principle, you can update the information here (eg. you can change the basis set) and re-submit. 
## If you do so, and everything works well, the code will compare the old and new qc_datas and if an update is detected...
## ...it will resubmit even if the same job under the old qc_data was flagged as complete.
## In other words, it should be save to update the qc_data, but I haven't tested all changes.

In [ ]:
## Currently, the code accepts a very limited set of qc_data options. Only Quantum Espresso and Gaussian codes are implemented, 
## together with a very limited set of functionals, basis sets, etc...

In [ ]:
## In Quantum Espresso, we have implemented different Pseudo-Potential Libraries available in the literature. 
## These are called in this section ('pseudo' variable). 
## See below for an example of the qc_data for Quantum Espresso:

#&qc_data
#   software     = 'qe'
#   version      = '7.0'
#   jobtype      = 'vc-relax'
#   functional   = 'pbe'
#   is_hubbard   = True
#   is_grimme    = True
#   grimme_type  = 'd3bj'
#   uterm        = 2.35
#   mix_beta     = 0.7
#   elec_conv    = 1e-5
#   pseudo       = 'vanderbilt'
#/

## 3) Run

In [ ]:
## Once the system and a job_file are prepared, we can submit the computation in the appropriate cluster
## To do so:

## 1) Configure the Cluster Environment using the "scope_config" script, installed by default: 

        ## The computation cluster should have Gaussian 16 installed, and a **configured environment**. 
        ## You can give it any name, but for the sake of simplicity, here we will call it "scope_env_cluster.npy" 

## 2) Upload the system file and the test_job.job to the cluster

        ## In principle, you should upload the system file to a dedicated folder inside the env.sys_path of the cluster enviromnent (env). 
        ## So, if env.sys_path = '/home/user/scope/tutorials/data/Tutorial_4/Systems', the system file should go to: '/home/user/scope/tutorials/data/Tutorial_4/Systems/water/water.npy'
        
        ## In practice, you can upload the system file anywhere in the cluster. 
        ## However, the cluster's Environment will force the system to be saved in the expected path: '//home/user/scope/tutorials/data/Tutorial_4/Systems/water/water.npy'
        ## This is the price to pay for the Environment controlling the paths

        ## The job_file can be saved anywhere. 

## 3) Then, you can use the 'scope_run_job' script.

        ## If the paths are the expected ones, you should type: 

```bash
scope_run_job -n scope_env_cluster.npy -s Systems/water/water.npy -j test_job.job
```
```

In [5]:
## The environment of the computation cluster (scope_env_cluster.npy in the example) will automatically update the paths stored in the system, and execute the job defined in test_job.
## If everything worked well, then you should have a gaussian16 job running in your cluster. 
## Whenever it finishes, just execute the same command above. Then, scope will register your job

## 4) Register

In [6]:
## When executing "scope_run_job", the script takes the indicated (in the job_file) Branch of the system, goes through the computational workflow...
## ... and takes the appropriate actions: 
 
## 1) If the job has not been submitted, it creates the input and submission file and does the submission.
## 2) If the job is running, it does nothing
## 3) If the job has finished, it reads the output and decides whether it finished 'well'
##   3.1) If not, it takes appropriate action. For instance, 
##      3.1.1) If it is a GEOMETRY OPTIMIZATION that didn't reach a zero-gradient solution, it creates and submits a new computation that continues the previous run
##      3.1.2) If it is an SCF computation in Quantum Espresso that didn't converge, it will modify some parameters and resubmit.
##      3.1.3) If the computation was aborted by SLURM or for some other reason, the logfile will be discarded, and the computation will be resubmitted.
##   3.2) If the job finished well, the logfile is parsed, and the relevant data is stored in the final state (fstate) defined by the computation
##        The job will be considered done, and even if you resubmit the same command, the computations won't run again. 

In [ ]:
## In tutorial number 6, we will see how those computations are "REGISTERED"